## ___Updating the photosynthetic pathways___
--------------------

In [1]:
!python --version

Python 3.13.8


The system cannot find the path specified.


In [2]:
import numpy as np
import pandas as pd

In [3]:
# https://www.tern.org.au/news/news-photosynthetic-pathways/
# https://portal.tern.org.au/metadata/TERN/1e16257d-57ae-48dd-bbad-5701e72a9f6d
# TERN is an Australia specific dataset
tern = pd.read_csv(r"../../data/chapter2/TERN/Photosynthetic_Pathways_of_Plants_TERN_v2_19092024.csv", encoding="latin1", 
        usecols=["genus", "speciesEpithet", "family", "photosyntheticPathway_confirmed", "photosyntheticPathway_inferred", "photosyntheticPathway_combined"], na_values='U')#, index_col=["genus", "speciesEpithet"])
tern_meta = pd.read_excel(r"../../data/chapter2/TERN/metadata_Photosynthetic_Pathways_of_Plants_TERN_v2_2024.xlsx", sheet_name="Data_Descriptor")

# extra spaces
tern.loc[:, "genus"] = tern.genus.str.strip()
tern.loc[:, "speciesEpithet"] = tern.speciesEpithet.str.strip()
tern.insert(loc=0, column="binominal", value=tern.genus.str.strip() + ' ' + tern.speciesEpithet.str.strip())

# in TRY, photosynthesis pathway is trait id 22
try_photo = pd.read_csv(r"../../data/chapter2/TRY/photosynthetic_pathways.txt", delimiter='\t', encoding="latin1", low_memory=False, decimal='.', usecols=["Dataset", "SpeciesName",
                        "AccSpeciesName", "OrigValueStr", "TraitID"]).dropna(subset=["AccSpeciesName", "OrigValueStr", "TraitID"]) # records where TraitId is NaN are mostly messy metadata rows

# extra spaces 
try_photo.loc[:, "SpeciesName"] = try_photo.SpeciesName.str.strip()
try_photo.loc[:, "OrigValueStr"] = try_photo.OrigValueStr.str.upper().str.replace('.', '')
# messy photosynthetic pathway information
PHOTOSYNTHETIC_PATHWAY_TYPES = { # try to make this as less messy as possible!!
    "C3": "C3",
    "C3?": "C3?",
    "C4": "C4",
    "C4?": "C4?",
    "CAM": "CAM",
    "CAM?": "CAM?",
    "C3/C4": "C3/C4",
    "C3C4": "C3/C4",
    "C3/CAM": "C3/CAM",
    "C3-CAM": "C3/CAM",
    "C4/CAM": "C4/CAM",
    "C4-CAM": "C4/CAM",
    "C3/C4/CAM": "C3/C4/CAM",
    "3": "C3"
}

try_photo.loc[:, "OrigValueStr"] = try_photo.OrigValueStr.replace(PHOTOSYNTHETIC_PATHWAY_TYPES) # clean up the irregularities in the column
subset_categorical = pd.read_csv(r"../../data/chapter2/FREDv3subset/FRED_subset_categorical.csv")

fred_photo = pd.read_csv(r"../../data/chapter2/FRED/FRED3_Entire_Database_2021.csv", low_memory=False, header=0, skiprows=range(1, 10), encoding="latin1",
                         usecols=("F01286", "F01287", "F00043", "F00004")).dropna(subset=("F01286", "F01287", "F00043")).drop_duplicates()
fred_photo.insert(loc=0, column="binominal", value=fred_photo.F01286.str.strip().str.capitalize() + ' ' + fred_photo.F01287.str.strip().str.lower())

# GRoot was not considered for mycorrhizal states as most of its mycorrhizal state info came from FungalRoot which was obe of the dataset that was used in populating the missing fields
groot = pd.read_csv(r"../../data/chapter2/GRooTFullVersion.csv", encoding="latin1", low_memory=False)
groot.insert(loc=0, column="binominal", value=groot.genus.str.capitalize().str.strip() + ' ' + groot.species.str.lower().str.strip())

In [4]:
species_of_interest = subset_categorical.binominal.drop_duplicates()
species_of_interest

0         Populus trichocarpa
1             Populus tremula
2            Altingia obovata
3       Cryptocarya chinensis
4      Elaeocarpus sylvestris
                ...          
207              Quercus alba
208             Quercus rubra
210       Pleioblastus amarus
234            Nyssa aquatica
235       Platanus acerifolia
Name: binominal, Length: 203, dtype: object

In [5]:
tern.query("binominal.isin(@species_of_interest)") # that's disappointing

,binominal,genus,speciesEpithet,family,photosyntheticPathway_confirmed,photosyntheticPathway_inferred,photosyntheticPathway_combined
250,Acacia auriculiformis,Acacia,auriculiformis,Fabaceae,C3,NaN,C3
2179,Acacia crassicarpa,Acacia,crassicarpa,Fabaceae,NaN,C3,C3


In [6]:
fred_photo.query("binominal.isin(@species_of_interest)").drop_duplicates() # that's nice

,binominal,F00004,F01286,F01287,F00043
0,Dicranopteris linearis,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Dicranopteris,linearis,C3
3,Cunninghamia lanceolata,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Cunninghamia,lanceolata,C3
6,Magnolia baillonii,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Magnolia,baillonii,C3
11,Acacia auriculiformis,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Acacia,auriculiformis,C3
15,Gordonia axillaris,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Gordonia,axillaris,C3
...,...,...,...,...,...
54946,Fraxinus mandshurica,"Wang Y, Gao G, Wang N, Wang Z, Gu J. 2019. Eff...",Fraxinus,mandshurica,C3
54952,Juglans mandshurica,"Wang Y, Gao G, Wang N, Wang Z, Gu J. 2019. Eff...",Juglans,mandshurica,C3
54964,Larix gmelinii,"Wang Y, Gao G, Wang N, Wang Z, Gu J. 2019. Eff...",Larix,gmelinii,C3
55060,Fraxinus mandshurica,"Liu YK, Fan C, Li XW, Ling YH, Zhou YG, Feng M...",Fraxinus,mandshurica,C3


In [7]:
# how many different pathways are there???
fred_photo.query("binominal.isin(@species_of_interest)").drop_duplicates().F00043.unique()

array(['C3'], dtype=object)

In [8]:
try_photo.query("AccSpeciesName.isin(@species_of_interest)").drop_duplicates()

,Dataset,SpeciesName,AccSpeciesName,TraitID,OrigValueStr
645,Sheffield-Iran-Spain Database,Acer pseudoplatanus,Acer pseudoplatanus,22.0,C3
1529,Sheffield-Iran-Spain Database,Fraxinus excelsior,Fraxinus excelsior,22.0,C3
2685,Sheffield-Iran-Spain Database,Sorbus aucuparia,Sorbus aucuparia,22.0,C3
4353,Abisko & Sheffield Database,Matteuccia struthiopteris,Matteuccia struthiopteris,22.0,C3
4605,Abisko & Sheffield Database,Populus tremula,Populus tremula,22.0,C3
...,...,...,...,...,...
2172385,TRY Categorical Traits Dataset (update 2018),Pinus tabuliformis,Pinus tabuliformis,22.0,C3
2178023,TRY Categorical Traits Dataset (update 2018),Styphnolobium japonicum,Styphnolobium japonicum,22.0,C3
2178734,TRY Categorical Traits Dataset (update 2018),Pithecellobium angulatum,Archidendron clypearia,22.0,C3
2181528,TRY Categorical Traits Dataset (update 2018),Millettia leptobotrys,Millettia leptobotrya,22.0,C3


In [10]:
try_photo.query("AccSpeciesName.isin(@species_of_interest)").drop_duplicates().OrigValueStr.unique()

array(['C3', 'C4', 'C3?', 'UNKNOWN'], dtype=object)

In [19]:
groot.query("not photosyntheticPathway.isna() and binominal.isin(@species_of_interest)").loc[:, ["binominal", "photosyntheticPathway", "references", "referencesDataset"]].drop_duplicates()

,binominal,photosyntheticPathway,references,referencesDataset
11,Acer saccharum,C3,"Aber JD, Melillo JM, Nadelhoffer KJ, McClaughe...","Nadelhoffer KJ, Raich JW. 1992. Fine root prod..."
18,Pinus strobus,C3,"Aber JD, Melillo JM, Nadelhoffer KJ, McClaughe...","Nadelhoffer KJ, Raich JW. 1992. Fine root prod..."
21,Quercus alba,C3,"Aber JD, Melillo JM, Nadelhoffer KJ, McClaughe...","Nadelhoffer KJ, Raich JW. 1992. Fine root prod..."
24,Quercus rubra,C3,"Aber JD, Melillo JM, Nadelhoffer KJ, McClaughe...","Nadelhoffer KJ, Raich JW. 1992. Fine root prod..."
30,Acer saccharum,C3,"Aber JD, Melillo JM, Nadelhoffer KJ, McClaughe...","Gill, R., and R. B. Jackson. 2003. Global Dist..."
...,...,...,...,...
112744,Acacia mangium,C3,Ziska et al. 1991 . Oecologia 86: 383-389,"Poorter H, Niinemets Ü, Walter A, Fiorani F, S..."
113188,Cystopteris sudetica,C3,Klimeová J. & Klime L. Clo-Pla3  database o...,NaN
113258,Equisetum hyemale,C3,Klimeová J. & Klime L. Clo-Pla3  database o...,NaN
113260,Equisetum pratense,C3,Klimeová J. & Klime L. Clo-Pla3  database o...,NaN


In [20]:
groot.query("not photosyntheticPathway.isna() and binominal.isin(@species_of_interest)").loc[:, ["binominal", "photosyntheticPathway", "references", "referencesDataset"]].drop_duplicates().photosyntheticPathway.unique()

array(['C3', 'C3/C4'], dtype=object)

In [22]:
# let's serialize these subsets
# extracts for photosynthetic pathways will have a _p postfix
fred_photo.query("binominal.isin(@species_of_interest)").drop_duplicates().to_csv(r"../../data/chapter2/extracts/fred_p.csv", index=False)
try_photo.query("AccSpeciesName.isin(@species_of_interest)").drop_duplicates().to_csv(r"../../data/chapter2/extracts/try_p.csv", index=False)
groot.query("not photosyntheticPathway.isna() and binominal.isin(@species_of_interest)").loc[:, ["binominal", "photosyntheticPathway", "references", "referencesDataset"]].\
            drop_duplicates().drop_duplicates().to_csv(r"../../data/chapter2/extracts/groot_p.csv", index=False)